<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Accessing L2 data (CALLIPSO)**

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)
2. <font size="3"> Generate a Bearer Token.
3. <font size="3"> Get Notebook from NASA-tutorial Github repository
4. <font size="3"> Get `environment.yml` file and install conda environment to run notebook.


 <span style='color:#ff6666'><font size="5"> **Objectives**
### Basics
- <font size="3"> Brief Introduction to <span style='color:#ff6666'>**OPeNDAP**</span> (i.e. **dap2** vs **dap4**). 
- <font size="3"> How to find <span style='color:#ff6666'>**OPeNDAP**</span> URLs.
- <font size="3"> Inspecting metadata (differences between **browser** / <span style='color:#ff6666'>**pydap**</span> and **Xarray**).
- <font size="3"> **Naive approach**: access data from a url using **Xarray** + <span style='color:#ff6666'>**pydap**<span style='color:black'>. Here we demonstrate different ways to authenticate.

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3"> **a)** Naive approaches.
- <font size="3"> **b)** Streaming data

## Appendix: Using curl


In [1]:
import os, sys
if sys.platform.startswith("linux"):
    # fattrs is missing even though it is installed. issue with libraries
    # 1. Remove system HDF5 from library lookup path
    os.environ.pop("LD_LIBRARY_PATH", None)
    # 2. Prepend conda environment's lib & bin paths
    conda_lib = sys.exec_prefix + "/lib"
    conda_bin = sys.exec_prefix + "/bin"
    os.environ["PATH"] = conda_bin + ":" + os.environ["PATH"]
    os.environ["LD_LIBRARY_PATH"] = conda_lib
    
import warnings
warnings.filterwarnings('ignore')


In [5]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import requests
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.net import create_session
from pydap.client import get_cmr_urls, consolidate_metadata, open_url
from pydap.client import to_netcdf as dap_to_netcdf

In [3]:
pydap.__version__

'3.5.9.dev41+g436dd71c5'

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

In [6]:
Callipso_L2_ccid = "C3463063995-LARC_CLOUD" # 
time_range = [dt.datetime(2023, 5, 15), dt.datetime(2023, 5, 20)] # One month of data

cmr_urls = get_cmr_urls(ccid=Callipso_L2_ccid, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

################################################ 
 We found a total of  117 OPeNDAP URLS!!!
################################################


In [7]:
# def get_opendap_urls(concept_id, time_range, _session=None):
#     """
#     Queries NASA's `Common Metadata Repository` to identify all OPeNDAP URLS
#     given collection concept ID and temporal time range.
#     """
#     cmr_url = 'https://cmr.earthdata.nasa.gov/search/granules'
#     if not _session:
#         _session = requests.Session() 
#     cmr_response = _session.get(cmr_url, params={'concept_id': concept_id,'temporal': time_range,'page_size': 500}, headers={'Accept': 'application/json'})
#     granules = cmr_response.json()['feed']['entry']
#     granules_urls = []
    
#     # Filter and only retain the OPeNDAP URLs
#     for granule in granules:
#         item = next((item['href'] for item in granule['links'] if "opendap" in item["href"]), None)
#         if item != None:
#             granules_urls.append(item)
#     return granules_urls

# start_date =  dt.datetime(2023, 5, 15)
# end_date =dt.datetime(2023, 5, 20)

# print(start_date, end_date,sep='\n')

# dt_format = '%Y-%m-%dT%H:%M:%SZ' # format requirement for datetime search
# temporal_str = start_date.strftime(dt_format) + ',' + end_date.strftime(dt_format)

# cmr_urls = get_opendap_urls(Callipso_L2_ccid, temporal_str)
# print(len(cmr_urls))

In [ ]:
cmr_urls[0]

# Inspecting Metadata

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Understanding DAP4**

<div style="text-align:center">
<img src="figures/DAP4vsDAP2.png" alt="drawing" width="750"/>    
</div>


### Basic Data Exploration

<font size="3.5">To inspect the metadata of a remote OPeNDAP URL (variables in the file), you :

* <font size="3.5"> Append a `.dmr` at the end of the URL
* <font size="3.5"> Open on a browser.
* <font size="3.5"> Useful when interested in subseting by variables for example.

For example:
    
https://opendap.earthdata.nasa.gov/collections/C3463063995-LARC_CLOUD/granules/CAL_LID_L2_01kmCLay-Standard-V5-00.2023-05-15T23-26-01ZD.hdf



<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [8]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = create_session(session=auth.get_session())
# my_session = requests.session()

# Accessing Metadata

<font size="3.5"> What are some tools, their differences, and what do they do.

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **PYDAP: Metadata-Only**



```{note}
Q: How do we tell the server which protocol to use?
A: By replaing the http -> dap4 in the URL
```


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)

In [ ]:
pyds.tree()

In [9]:
nbytes = []
for url in cmr_urls:
    pyds = open_url(url, protocol="dap4", session=my_session)
    nbytes.append(pyds.nbytes)

In [1]:
sum(nbytes)

NameError: name 'nbytes' is not defined

<span style='font-family:serif'> <font size="5"><span style='color:#0066cc'> **Recommended Worklow for Streaming Data**


* <font size="3.5"><span style='color:#0066cc'> **Data Exploration**</span>: `Xarray + PyDAP`. When interested in visualizing one or two variables. Identify regions of interest (interactively).

However, if the goal is to stream data via OPeNDAP, Xarray+opendap requires extra logic to get the **best performance** that is stil subpar compared to **talking to the server directly**


* <font size="3.5"><span style='color:#0066cc'> **Streaming a Data Subset**</span>: Pure PyDAP to parallelize direct server (subset) requests. See below:




In [13]:
output_path = "/home/idies/workspace/Temporary/Mikejmnez/scratch/data/"

# keep_variables = ["Lidar_Data_Altitudes", "Profile_ID", "Latitude", "Longitude",
#                   "Profile_Time", "Profile_UTC_Time", "Tropopause_Height", "Tropopause_Temperature", 
#                   "Ocean_Derived_Column_Optical_Depth", "Lidar_Surface_Detection"]
keep_variables = [
    '/Lidar_Surface_Detection', "/Ocean_Derived_Column_Optical_Depth",
    "/Lidar_Data_Altitudes", "/Profile_ID", "/Latitude", "/Longitude", 
    "/Profile_Time", "/Profile_UTC_Time", "/Day_Night_Flag", "/Tropopause_Height", 
    "/Tropopause_Temperature","/Snow_Ice_Surface_Type", "/Opacity_Flag", 
]
dap4ce = "?dap4.ce="+";".join(keep_variables)
print(dap4ce)

?dap4.ce=/Lidar_Surface_Detection;/Ocean_Derived_Column_Optical_Depth;/Lidar_Data_Altitudes;/Profile_ID;/Latitude;/Longitude;/Profile_Time;/Profile_UTC_Time;/Day_Night_Flag;/Tropopause_Height;/Tropopause_Temperature;/Snow_Ice_Surface_Type;/Opacity_Flag


In [ ]:
cmr_urls[0]

In [ ]:
cmr_urls[1]

# Single File

### OPeNDAP + pydap-only logic


### metadata only

In [ ]:
%%time
pyds = open_url(cmr_urls[0].replace("https", "dap4")+dap4ce, session=my_session)

In [ ]:
pyds.tree()

In [ ]:
%%time
to_netcdf(cmr_urls[0], session=my_session, keep_variables=keep_variables, output_path=output_path)

In [ ]:
# dT = xr.open_datatree(output_path+"/CAL_LID_L2_01kmCLay-Standard-V5-00.2023-05-15T23-26-01ZD.nc4")

In [ ]:
# dT

### xarray approach

- 2-step process: 
1. Create an Xarray Dataset object.
2. Download as file of choice.


In [ ]:
%%time
ds = xr.open_datatree(cmr_urls[0].replace("https", "dap4")+dap4ce, session=my_session, engine='pydap')

In [ ]:
%%time
ds.to_netcdf(output_path+'/single_download.nc4')

### Multiple downloads

- OPenDAP & pydap-only approach

In [14]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, keep_variables=keep_variables, output_path=output_path)

CPU times: user 161 ms, sys: 30.5 ms, total: 191 ms
Wall time: 1min 43s


## check the data

In [16]:
ds = xr.open_datatree(output_path+"CAL_LID_L2_01kmCLay-Standard-V5-00.2023-05-17T15-41-01ZN.nc4")

In [20]:
from pathlib import Path

data_dir = Path(output_path)  # <-- change this
files = sorted(data_dir.glob("*.nc4"))  